In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e4/train.csv
/kaggle/input/competitions/playground-series-s6e4/test.csv


In [2]:
# =========================================
# 🔥 CONFIG
# =========================================

N_TRIALS = 10
N_TRIALS_META = 10
N_TRIALS_BLEND = 10
PSEUDO_THRESHOLD = 0.80
N_SPLITS = 5
RANDOM_STATE = 42

DATA_PATH = "/kaggle/input/competitions/playground-series-s6e4/"

# =========================================
# 🔥 IMPORTS
# =========================================

import numpy as np
import pandas as pd
import optuna
import shap

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

print("✅ Imports done")

# =========================================
# 🔥 LOAD DATA
# =========================================

train = pd.read_csv(DATA_PATH + "train.csv")
test = pd.read_csv(DATA_PATH + "test.csv")

target_col = train.columns[-1]

y = train[target_col]
X = train.drop(columns=[target_col])
X_test = test.copy()

# =========================================
# 🔥 FEATURE ENGINEERING (STRONG)
# =========================================

def create_features(df):
    df = df.copy()

    df["evapo"] = df["Temperature_C"] * (1 - df["Humidity"]/100)
    df["water_deficit"] = df["evapo"] - df["Rainfall_mm"]
    df["stress"] = df["Temperature_C"] / (df["Soil_Moisture"] + 1)

    df["moisture_rain_ratio"] = df["Soil_Moisture"] / (df["Rainfall_mm"] + 1)
    df["rain_temp_ratio"] = df["Rainfall_mm"] / (df["Temperature_C"] + 1)

    df["temp_humidity"] = df["Temperature_C"] * df["Humidity"]
    df["rain_moisture"] = df["Rainfall_mm"] * df["Soil_Moisture"]

    df["temp_squared"] = df["Temperature_C"] ** 2
    df["rain_log"] = np.log1p(df["Rainfall_mm"])

    df["wind_sun"] = df["Wind_Speed_kmh"] * df["Sunlight_Hours"]
    df["temp_sun"] = df["Temperature_C"] * df["Sunlight_Hours"]

    df["crop_stage"] = df["Crop_Type"].astype(str) + "_" + df["Crop_Growth_Stage"].astype(str)
    df["region_season"] = df["Region"].astype(str) + "_" + df["Season"].astype(str)

    df["crop_water"] = df["Crop_Type"].astype(str) + "_" + df["Water_Source"].astype(str)
    df["soil_crop"] = df["Soil_Type"].astype(str) + "_" + df["Crop_Type"].astype(str)

    return df

X = create_features(X)
X_test = create_features(X_test)

# =========================================
# 🔥 ENCODING
# =========================================

for col in X.select_dtypes("object").columns:
    X[col] = X[col].astype("category")
    X_test[col] = X_test[col].astype("category")

cat_cols = X.select_dtypes("category").columns.tolist()

le = LabelEncoder()
y = le.fit_transform(y)
num_classes = len(np.unique(y))

# =========================================
# 🔥 SHAP FEATURE SELECTION
# =========================================

print("🚀 SHAP selection...")

sample_idx = np.random.choice(len(X), min(2000, len(X)), replace=False)
X_sample = X.iloc[sample_idx].copy()
y_sample = y[sample_idx]

for col in X_sample.select_dtypes("category").columns:
    X_sample[col] = X_sample[col].cat.codes

temp_model = xgb.XGBClassifier(n_estimators=120, tree_method="hist")
temp_model.fit(X_sample, y_sample)

explainer = shap.TreeExplainer(temp_model)
shap_vals = explainer.shap_values(X_sample, approximate=True)

if isinstance(shap_vals, list):
    shap_vals = np.stack(shap_vals, axis=-1)

importance = np.abs(shap_vals).mean(axis=(0,2))
keep_cols = X.columns[importance > np.percentile(importance, 20)]

X = X[keep_cols]
X_test = X_test[keep_cols]

cat_cols = [c for c in cat_cols if c in X.columns]

print("✅ Features:", len(keep_cols))

# =========================================
# 🔥 XGB SAFE DATA
# =========================================

X_xgb = X.copy()
X_test_xgb = X_test.copy()

for col in X_xgb.select_dtypes("category").columns:
    X_xgb[col] = X_xgb[col].cat.codes
    X_test_xgb[col] = X_test_xgb[col].cat.codes

# =========================================
# 🔥 CV
# =========================================

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# =========================================
# 🔥 OPTUNA BASE MODELS
# =========================================

def xgb_obj(trial):
    params = {
        "objective": "multi:softprob",
        "num_class": num_classes,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "n_estimators": 400,
        "tree_method": "hist"
    }
    scores=[]
    for tr,val in skf.split(X_xgb,y):
        m=xgb.XGBClassifier(**params)
        m.fit(X_xgb.iloc[tr],y[tr])
        scores.append(balanced_accuracy_score(y[val],m.predict(X_xgb.iloc[val])))
    return np.mean(scores)

def lgb_obj(trial):
    params={
        "objective":"multiclass",
        "num_class":num_classes,
        "learning_rate":trial.suggest_float("learning_rate",0.01,0.1),
        "num_leaves":trial.suggest_int("num_leaves",20,100),
        "n_estimators":400
    }
    scores=[]
    for tr,val in skf.split(X,y):
        m=lgb.LGBMClassifier(**params)
        m.fit(X.iloc[tr],y[tr])
        scores.append(balanced_accuracy_score(y[val],m.predict(X.iloc[val])))
    return np.mean(scores)

def cat_obj(trial):
    params={
        "iterations":400,
        "learning_rate":trial.suggest_float("learning_rate",0.01,0.1),
        "depth":trial.suggest_int("depth",3,8),
        "verbose":0
    }
    scores=[]
    for tr,val in skf.split(X,y):
        m=CatBoostClassifier(**params)
        m.fit(X.iloc[tr],y[tr],cat_features=cat_cols)
        scores.append(balanced_accuracy_score(y[val],m.predict(X.iloc[val])))
    return np.mean(scores)

def et_obj(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators",200,500),
        "max_depth": trial.suggest_int("max_depth",6,20),
        "min_samples_split": trial.suggest_int("min_samples_split",2,10),
        "n_jobs": -1,
        "random_state": RANDOM_STATE
    }

    scores=[]
    for tr,val in skf.split(X_xgb,y):
        m = ExtraTreesClassifier(**params)
        m.fit(X_xgb.iloc[tr],y[tr])
        scores.append(balanced_accuracy_score(y[val],m.predict(X_xgb.iloc[val])))
    return np.mean(scores)

print("🚀 tuning models")



study_xgb=optuna.create_study(direction="maximize")
study_xgb.optimize(xgb_obj,n_trials=N_TRIALS)

study_lgb=optuna.create_study(direction="maximize")
study_lgb.optimize(lgb_obj,n_trials=N_TRIALS)

# study_cat=optuna.create_study(direction="maximize")
# study_cat.optimize(cat_obj,n_trials=N_TRIALS)

study_et = optuna.create_study(direction="maximize")
study_et.optimize(et_obj, n_trials=N_TRIALS)

# =========================================
# 🔥 OOF STACKING
# =========================================

oof_xgb=np.zeros((len(X),num_classes))
oof_lgb=np.zeros((len(X),num_classes))
# oof_cat=np.zeros((len(X),num_classes))
oof_et  = np.zeros((len(X),num_classes))

for tr,val in skf.split(X,y):

    xgb_m=xgb.XGBClassifier(**study_xgb.best_params,n_estimators=400)
    lgb_m=lgb.LGBMClassifier(**study_lgb.best_params,n_estimators=400)
    # cat_m=CatBoostClassifier(**study_cat.best_params,iterations=400,verbose=0)
    et_m  = ExtraTreesClassifier(**study_et.best_params)

    xgb_m.fit(X_xgb.iloc[tr],y[tr])
    lgb_m.fit(X.iloc[tr],y[tr])
    # cat_m.fit(X.iloc[tr],y[tr],cat_features=cat_cols)
    et_m.fit(X_xgb.iloc[tr],y[tr])

    oof_xgb[val]=xgb_m.predict_proba(X_xgb.iloc[val])
    oof_lgb[val]=lgb_m.predict_proba(X.iloc[val])
    # oof_cat[val]=cat_m.predict_proba(X.iloc[val])
    oof_et[val]=et_m.predict_proba(X_xgb.iloc[val])

# X_meta=np.hstack([oof_xgb,oof_lgb,oof_cat])
X_meta = np.hstack([oof_xgb,oof_lgb,oof_et])

# =========================================
# 🔥 META OPTUNA
# =========================================

def meta_obj(trial):
    C=trial.suggest_float("C",0.01,10)
    m=LogisticRegression(C=C,max_iter=1000)
    scores=[]
    for tr,val in skf.split(X_meta,y):
        m.fit(X_meta[tr],y[tr])
        scores.append(balanced_accuracy_score(y[val],m.predict(X_meta[val])))
    return np.mean(scores)

study_meta=optuna.create_study(direction="maximize")
study_meta.optimize(meta_obj,n_trials=N_TRIALS_META)

meta=LogisticRegression(**study_meta.best_params,max_iter=1000)
meta.fit(X_meta,y)

# =========================================
# 🔥 FINAL TRAIN
# =========================================

xgb_f=xgb.XGBClassifier(**study_xgb.best_params,n_estimators=600)
lgb_f=lgb.LGBMClassifier(**study_lgb.best_params,n_estimators=600)
# cat_f=CatBoostClassifier(**study_cat.best_params,iterations=600,verbose=0)
et_f  = ExtraTreesClassifier(**study_et.best_params)

xgb_f.fit(X_xgb,y)
lgb_f.fit(X,y)
# cat_f.fit(X,y,cat_features=cat_cols)
et_f.fit(X_xgb,y)

test_xgb=xgb_f.predict_proba(X_test_xgb)
test_lgb=lgb_f.predict_proba(X_test)
# test_cat=cat_f.predict_proba(X_test)
test_et  = et_f.predict_proba(X_test_xgb)

# stack_test=np.hstack([test_xgb,test_lgb,test_cat])
stack_test = np.hstack([test_xgb,test_lgb,test_et])
stack_preds=meta.predict_proba(stack_test)

# =========================================
# 🔥 BLENDING OPTUNA
# =========================================

def blend_obj(trial):
    w1=trial.suggest_float("xgb",0,1)
    w2=trial.suggest_float("lgb",0,1)
    # w3=trial.suggest_float("cat",0,1)
    w3=trial.suggest_float("et",0,1)
    w4=trial.suggest_float("meta",0,1)

    wsum=w1+w2+w3+w4
    w1,w2,w3,w4=[w/wsum for w in [w1,w2,w3,w4]]

    # blend=w1*oof_xgb+w2*oof_lgb+w3*oof_cat+w4*meta.predict_proba(X_meta)
    blend=w1*oof_xgb+w2*oof_lgb+w3*oof_et+w4*meta.predict_proba(X_meta)
    
    preds=np.argmax(blend,axis=1)
    return balanced_accuracy_score(y,preds)

study_blend=optuna.create_study(direction="maximize")
study_blend.optimize(blend_obj,n_trials=N_TRIALS_BLEND)

w=study_blend.best_params
wsum=sum(w.values())
w={k:v/wsum for k,v in w.items()}

final_probs=(w["xgb"]*test_xgb+
             w["lgb"]*test_lgb+
             # w["cat"]*test_cat+
             w["et"]*test_et+
             w["meta"]*stack_preds)

# =========================================
# 🔥 PSEUDO LABELING (SAFE)
# =========================================

conf=np.max(final_probs,axis=1)
mask=conf>PSEUDO_THRESHOLD

if mask.sum()>0:
    print("Pseudo samples:",mask.sum())

    X_aug=pd.concat([X,X_test[mask]])
    y_aug=np.concatenate([y,np.argmax(final_probs[mask],axis=1)])

    xgb_f.fit(pd.concat([X_xgb,X_test_xgb[mask]]),y_aug)

# =========================================
# 🔥 FINAL OUTPUT
# =========================================

final_preds=le.inverse_transform(np.argmax(final_probs,axis=1))

sub=pd.read_csv(DATA_PATH+"sample_submission.csv")
sub[target_col]=final_preds
sub.to_csv("submission.csv",index=False)

print("🚀 FINAL SUBMISSION READY")

✅ Imports done
🚀 SHAP selection...


[I 2026-04-22 06:12:31,797] A new study created in memory with name: no-name-9e1b2a0c-c2e8-4bcb-8850-1e44a79c5a8d


✅ Features: 28
🚀 tuning models


[I 2026-04-22 06:17:28,618] Trial 0 finished with value: 0.9616049598429113 and parameters: {'learning_rate': 0.03510367986601822, 'max_depth': 7, 'subsample': 0.6807424897169989, 'colsample_bytree': 0.7036701726756904}. Best is trial 0 with value: 0.9616049598429113.
[I 2026-04-22 06:22:17,554] Trial 1 finished with value: 0.9619215410581343 and parameters: {'learning_rate': 0.0747812907972694, 'max_depth': 7, 'subsample': 0.6394313962684601, 'colsample_bytree': 0.7816903959040882}. Best is trial 1 with value: 0.9619215410581343.
[I 2026-04-22 06:26:59,795] Trial 2 finished with value: 0.9620317825191073 and parameters: {'learning_rate': 0.07372699069377239, 'max_depth': 7, 'subsample': 0.9313744786067147, 'colsample_bytree': 0.9894390600528163}. Best is trial 2 with value: 0.9620317825191073.
[I 2026-04-22 06:30:44,451] Trial 3 finished with value: 0.9616515716621123 and parameters: {'learning_rate': 0.06500731693670612, 'max_depth': 5, 'subsample': 0.7103748554943142, 'colsample_byt

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.099890 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5319
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 28
[LightGBM] [Info] Start training from score -3.400721
[LightGBM] [Info] Start training from score -0.532443
[LightGBM] [Info] Start training from score -0.968948
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.023239 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5320
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 28
[LightGBM] [Info] Start training from score -3.400781
[LightGBM] [Info] Start training from score -0.532443
[LightGBM] [Info] Start training from score -0.968943
[LightGBM] [Info] Auto-choosing 

...

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.022712 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5319
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 28
[LightGBM] [Info] Start training from score -3.400721
[LightGBM] [Info] Start training from score -0.532443
[LightGBM] [Info] Start training from score -0.968948
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.023143 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5320
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 28
[LightGBM] [Info] Start training from score -3.400781
[LightGBM] [Info] Start training from score -0.532443
[LightGBM] [Info] Star

[I 2026-04-22 09:53:24,356] A new study created in memory with name: no-name-b0a0eeb5-82d6-4200-8bc7-b54dc6538c36
[I 2026-04-22 09:53:39,231] Trial 0 finished with value: 0.9619459037174606 and parameters: {'C': 8.648487749458127}. Best is trial 0 with value: 0.9619459037174606.
[I 2026-04-22 09:53:53,854] Trial 1 finished with value: 0.9619267527173816 and parameters: {'C': 3.757231631776734}. Best is trial 0 with value: 0.9619459037174606.
[I 2026-04-22 09:54:08,445] Trial 2 finished with value: 0.9619221619715379 and parameters: {'C': 4.656589178278979}. Best is trial 0 with value: 0.9619459037174606.
[I 2026-04-22 09:54:23,103] Trial 3 finished with value: 0.9619573766450975 and parameters: {'C': 0.9760783470477565}. Best is trial 3 with value: 0.9619573766450975.
[I 2026-04-22 09:54:38,129] Trial 4 finished with value: 0.9619118698428124 and parameters: {'C': 3.4857640752424195}. Best is trial 3 with value: 0.9619573766450975.
[I 2026-04-22 09:54:52,661] Trial 5 finished with valu

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.028797 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5318
[LightGBM] [Info] Number of data points in the train set: 630000, number of used features: 28
[LightGBM] [Info] Start training from score -3.400769
[LightGBM] [Info] Start training from score -0.532441
[LightGBM] [Info] Start training from score -0.968947


[I 2026-04-22 10:07:55,098] A new study created in memory with name: no-name-bd9e990e-2343-40bf-b139-9a54a7e64117
[I 2026-04-22 10:07:55,299] Trial 0 finished with value: 0.9619931782826784 and parameters: {'xgb': 0.4640346020025927, 'lgb': 0.026214659771428228, 'et': 0.22732420207544624, 'meta': 0.5671706636083071}. Best is trial 0 with value: 0.9619931782826784.
[I 2026-04-22 10:07:55,482] Trial 1 finished with value: 0.9615835276501237 and parameters: {'xgb': 0.04435822953760127, 'lgb': 0.24145773005728566, 'et': 0.26090968355632016, 'meta': 0.15029177759886636}. Best is trial 0 with value: 0.9619931782826784.
[I 2026-04-22 10:07:55,664] Trial 2 finished with value: 0.9619205655553484 and parameters: {'xgb': 0.14729396456396615, 'lgb': 0.6118382212179607, 'et': 0.6562942900287254, 'meta': 0.9932817079640976}. Best is trial 0 with value: 0.9619931782826784.
[I 2026-04-22 10:07:55,843] Trial 3 finished with value: 0.9617681022134769 and parameters: {'xgb': 0.6804485968864684, 'lgb': 0

Pseudo samples: 268257
🚀 FINAL SUBMISSION READY
